In [ ]:
%pip install Wikipedia-API
%pip install pandas
%pip install beautifulsoup4
%pip install requests
%pip install urllib3
%pip install networkx
%pip install matplotlib
%pip install scipy


In [8]:
import wikipediaapi

In [9]:
wiki = wikipediaapi.Wikipedia(user_agent='mental-health-wiki-network (pprajap7@terpmail.umd.edu)', language='en')


In [13]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import unquote, quote

API = "https://en.wikipedia.org/w/api.php"
session = requests.Session()
session.headers.update({"User-Agent": "mental-health-wiki-network (pprajap7@terpmail.umd.edu)"})

def api_get(params):
    params = {**params, "format": "json", "formatversion": 2}
    r = session.get(API, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [ ]:
# helper functions for extracting links from a page and getting category members

REMOVE_SELECTORS = [
    "div.navbox", "table.navbox", "div.navbox-styles", "table.sidebar",
    "table.vertical-navbox", "div.reflist", "ol.references", "div.refbegin",
    "table.metadata", "div.hatnote",
]
NS_PREFIXES = ("File:", "Category:", "Help:", "Wikipedia:", "Template:", "Special:",
               "Portal:", "Talk:", "Template talk:", "Module:", "Draft:", "User:", "MOS:")

def get_page_html(title):
    data = api_get({"action": "parse", "page": title, "prop": "text", "redirects": 1})
    return None if "error" in data else data["parse"]["text"]

def clean_title(t):
    t = unquote(t).replace("_", " ").split("#")[0].strip()
    return t[:1].upper() + t[1:] if t else t

def extract_links(html):
    soup = BeautifulSoup(html, "html.parser")
    for sel in REMOVE_SELECTORS:
        for tag in soup.select(sel):
            tag.decompose()
    links = set()
    for a in soup.find_all("a", href=True):
        if a["href"].startswith("/wiki/"):
            title = clean_title(a["href"][len("/wiki/"):])
            if title and not title.startswith(NS_PREFIXES):
                links.add(title)
    return links

def get_category_members(category):
    titles = []
    params = {"action": "query", "list": "categorymembers", "cmtitle": category,
              "cmnamespace": 0, "cmlimit": "max"}
    while True:
        data = api_get(params)
        titles += [m["title"] for m in data["query"]["categorymembers"]]
        if "continue" not in data:
            break
        params.update(data["continue"])
    return titles

In [ ]:
# helper functions for resolving redirects and getting page lengths and pageviews

def resolve_titles(titles):
    """Return {original title: real title} and {real title: article length}."""
    mapping, lengths = {}, {}
    titles = list(titles)
    for i in range(0, len(titles), 50):
        batch = titles[i:i + 50]
        q = api_get({"action": "query", "titles": "|".join(batch),
                     "redirects": 1, "prop": "info"})["query"]
        norm = {n["from"]: n["to"] for n in q.get("normalized", [])}
        redir = {r["from"]: r["to"] for r in q.get("redirects", [])}
        for p in q["pages"]:
            if not p.get("missing") and p.get("ns") == 0:
                lengths[p["title"]] = p["length"]
        for t in batch:
            t2 = norm.get(t, t)
            final = redir.get(t2, t2)
            if final in lengths:
                mapping[t] = final
        time.sleep(0.1)
    return mapping, lengths

def get_redirect_aliases(titles):
    """Map each redirect that points to one of our articles -> that article."""
    aliases = {}
    titles = list(titles)
    for i in range(0, len(titles), 50):
        params = {"action": "query", "titles": "|".join(titles[i:i + 50]),
                  "prop": "redirects", "rdlimit": "max", "rdnamespace": 0}
        while True:
            data = api_get(params)
            for p in data["query"]["pages"]:
                for r in p.get("redirects", []):
                    aliases[r["title"]] = p["title"]
            if "continue" not in data:
                break
            params.update(data["continue"])
            time.sleep(0.1)
    return aliases

PV_URL = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
          "en.wikipedia/all-access/user/{title}/monthly/20251001/20260930")

def get_pageviews(title):
    r = session.get(PV_URL.format(title=quote(title.replace(" ", "_"), safe="")), timeout=30)
    return sum(i["views"] for i in r.json().get("items", [])) if r.status_code == 200 else 0

In [ ]:
# article set construction
disorder_titles = extract_links(get_page_html("List of mental disorders"))
disorder_titles |= {"Bipolar disorder", "Anorexia nervosa", "Bulimia nervosa"}  # added as they were missing in the first extraction

therapy_titles = set(get_category_members("Category:Psychotherapies"))
if not therapy_titles: 
    therapy_titles = extract_links(get_page_html("List of psychotherapies"))
print(len(disorder_titles), "links from the disorders list")
print(len(therapy_titles), "therapy links")

mapping, lengths = resolve_titles(disorder_titles | therapy_titles)

nodes = {}
for t in therapy_titles:
    if t in mapping:
        nodes[mapping[t]] = "therapy"
for t in disorder_titles:
    if t in mapping:
        nodes[mapping[t]] = "disorder"   


# updated after first extraction to remove references, organizations, substances, and other non-disorder/therapy articles
EXCLUDE = {
    "Mental disorder", "Diagnostic and Statistical Manual of Mental Disorders",
    "DSM-5", "DSM-IV", "ICD-10", "ICD-11", "International Classification of Diseases",
    "Psychiatry", "Psychology", "World Health Organization", "American Psychiatric Association",
    "Caffeine", "Cathinone", "Dissociative", "Effects of cannabis", "Hallucinogen",
    "Inhalant", "Nicotine", "Opioid", "Substituted amphetamine", "Synthetic cannabinoids",
    "Mental health professional", "Psychiatrist", "Classification of mental disorders",
    "ICD-11 classification of personality disorders", "Neurological disorder",
    "Non-rapid eye movement sleep", "Hypnagogia", "Hypnopompia",
    "Sadomasochism", "Sleep deprivation",
}
# additional exclusions after reviewing the list of articles
EXCLUDE |= {
    "Diana Fosha", "Frank Farrelly", "Fred Newman (philosopher)", "Gerda Boyesen",
    "Harvey Chochinov", "Ivan Boszormenyi-Nagy", "Janina Scarlet", "Vittorio Guidano",
    "Psychotherapy", "Treatment of mental disorders", "Services for mental disorders",
    "Positive psychology", "Humanistic psychology", "Transpersonal psychology",
    "Depth psychology", "Process psychology", "Somatic psychology",
    "Integral theory", "Object relations theory", "Transtheoretical model",
    "Abreaction", "Nonviolent Communication",
}
nodes = {t: typ for t, typ in nodes.items()
         if t not in EXCLUDE and not t.startswith(("List of", "Outline of"))}

seed_df = pd.DataFrame(sorted(nodes.items()), columns=["title", "type"])
print(len(seed_df), "articles in the final set")
print(seed_df["type"].value_counts())
pd.set_option("display.max_rows", None)
seed_df

In [ ]:
# 
out_links = {}
for i, title in enumerate(nodes, 1):
    try:
        html = get_page_html(title)
        out_links[title] = extract_links(html) if html else set()
    except Exception as e:
        print("Failed:", title, e)
        out_links[title] = set()
    if i % 50 == 0:
        print(f"{i}/{len(nodes)} articles done")
    time.sleep(0.1)
print("Done")

In [ ]:
aliases = get_redirect_aliases(nodes)

edges = set()
for source, links in out_links.items():
    for link in links:
        target = aliases.get(link, link)
        if target in nodes and target != source:
            edges.add((source, target))

links_df = pd.DataFrame(sorted(edges), columns=["source", "target"])
print(len(links_df), "edges")
links_df.head()

In [ ]:
rows = []
for title, typ in nodes.items():
    rows.append({
        "title": title,
        "type": typ,
        "length_bytes": lengths.get(title),
        "total_links_out": len(out_links[title]),
        "pageviews_12mo": get_pageviews(title),
    })
    time.sleep(0.05)

articles_df = pd.DataFrame(rows)
articles_df.to_csv("articles.csv", index=False)
links_df.to_csv("links.csv", index=False)

print(f"{len(articles_df)} articles, {len(links_df)} edges")
articles_df.sort_values("pageviews_12mo", ascending=False).head(10)

In [ ]:
## debugging missing links and pageviews

print((articles_df["total_links_out"] == 0).sum(), "articles with 0 links collected")
print((articles_df["pageviews_12mo"] == 0).sum(), "articles with 0 page views")
articles_df[articles_df["total_links_out"] == 0]["title"].tolist()

In [ ]:
def get_page_html_retry(title, tries=4):
    for attempt in range(tries):
        try:
            html = get_page_html(title)
            if html:
                return html
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {title}: {e}")
        time.sleep(2 ** attempt)  # wait 1, 2, 4, 8 seconds
    return None

def get_pageviews_retry(title, tries=4):
    url = PV_URL.format(title=quote(title.replace(" ", "_"), safe=""))
    for attempt in range(tries):
        r = session.get(url, timeout=30)
        if r.status_code == 200:
            return sum(i["views"] for i in r.json().get("items", []))
        if r.status_code == 404:  # no page view data exists
            return 0
        time.sleep(2 ** attempt)
    print("Page views failed for", title)
    return 0

# Retry link collection for articles with no links
missing = [t for t, links in out_links.items() if not links]
print(len(missing), "articles to retry for links")
for title in missing:
    html = get_page_html_retry(title)
    out_links[title] = extract_links(html) if html else set()
    time.sleep(0.5)
print("Still missing links:", [t for t in missing if not out_links[t]])

In [ ]:
aliases = get_redirect_aliases(nodes)

edges = set()
for source, links in out_links.items():
    for link in links:
        target = aliases.get(link, link)
        if target in nodes and target != source:
            edges.add((source, target))

links_df = pd.DataFrame(sorted(edges), columns=["source", "target"])
print(len(links_df), "edges")
links_df.head()

In [ ]:
for i, row in articles_df.iterrows():
    articles_df.at[i, "total_links_out"] = len(out_links[row["title"]])
    if row["pageviews_12mo"] == 0:
        articles_df.at[i, "pageviews_12mo"] = get_pageviews_retry(row["title"])
        time.sleep(0.2)

articles_df.to_csv("articles.csv", index=False)
links_df.to_csv("links.csv", index=False)

print((articles_df["total_links_out"] == 0).sum(), "articles still with 0 links")
print((articles_df["pageviews_12mo"] == 0).sum(), "articles still with 0 page views")
print(len(links_df), "edges")
articles_df.sort_values("pageviews_12mo", ascending=False).head(10)

In [ ]:
# graph construction and visualization
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

articles_df = pd.read_csv("articles.csv")
links_df = pd.read_csv("links.csv")

G = nx.DiGraph()
for _, row in articles_df.iterrows():
    G.add_node(row["title"], type=row["type"], pageviews=row["pageviews_12mo"])
G.add_edges_from(links_df[["source", "target"]].itertuples(index=False, name=None))

pagerank = nx.pagerank(G)
isolated = [n for n in G if G.degree(n) == 0]
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print(len(isolated), "articles with no links to or from other articles in the set")

In [ ]:
COLORS = {"disorder": "#4C72B0", "therapy": "#DD8452"}
H = G.subgraph([n for n in G if G.degree(n) > 0])
pos = nx.spring_layout(H, k=0.3, seed=42) 

max_pr = max(pagerank.values())
sizes = [30 + 900 * pagerank[n] / max_pr for n in H]
colors = [COLORS[H.nodes[n]["type"]] for n in H]
top10 = sorted(H, key=pagerank.get, reverse=True)[:10]

fig, ax = plt.subplots(figsize=(14, 12))
nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.06, width=0.5, arrows=False)
nx.draw_networkx_nodes(H, pos, ax=ax, node_size=sizes, node_color=colors, alpha=0.85)
nx.draw_networkx_labels(H, pos, labels={n: n for n in top10}, font_size=9, ax=ax,
                        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))

legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=t.title())
          for t, c in COLORS.items()]
ax.legend(handles=legend, loc="upper left", fontsize=11)
ax.set_title("Mental health articles on Wikipedia (node size = PageRank)", fontsize=14)
ax.axis("off")
plt.savefig("network_full.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# zoom into the 30 most central articles
top30 = sorted(G, key=pagerank.get, reverse=True)[:30]
S = G.subgraph(top30)
pos_s = nx.spring_layout(S, k=1.2, seed=42)

fig, ax = plt.subplots(figsize=(14, 11))
nx.draw_networkx_edges(S, pos_s, ax=ax, alpha=0.25, arrows=True, arrowsize=8,
                       node_size=[300 + 2500 * pagerank[n] / max_pr for n in S])
nx.draw_networkx_nodes(S, pos_s, ax=ax, node_color=[COLORS[S.nodes[n]["type"]] for n in S],
                       node_size=[300 + 2500 * pagerank[n] / max_pr for n in S], alpha=0.9)
nx.draw_networkx_labels(S, pos_s, font_size=8, ax=ax,
                        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))
ax.legend(handles=legend, loc="upper left", fontsize=11)
ax.set_title("The 30 most central mental health articles (node size = PageRank)", fontsize=14)
ax.axis("off")
plt.savefig("network_top30.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# important measures for every article by in-degree, PageRank

in_degree = dict(G.in_degree())

articles_df["in_degree"] = articles_df["title"].map(in_degree)
articles_df["pagerank"] = articles_df["title"].map(pagerank)

cols = ["title", "type", "in_degree", "pagerank", "pageviews_12mo"]
articles_df.sort_values("pagerank", ascending=False)[cols].head(10).round(4)

,title,type,in_degree,pagerank,pageviews_12mo
106,Cognitive behavioral therapy,therapy,158,0.0415,367944
288,Major depressive disorder,disorder,112,0.0260,632361
345,Post-traumatic stress disorder,disorder,81,0.0210,513241
373,Schizophrenia,disorder,72,0.0202,1227845
60,Psychoanalysis,therapy,68,0.0180,253073
390,Bipolar disorder,disorder,76,0.0174,1127187
296,Attention deficit hyperactivity disorder,disorder,65,0.0156,1455571
227,Psychosis,disorder,56,0.0151,725552
230,Autism,disorder,58,0.0143,1459626
372,Obsessive–compulsive disorder,disorder,56,0.0136,866966


In [ ]:
# top 10 for each measure

top10 = pd.DataFrame({
    "In-degree": articles_df.nlargest(10, "in_degree")["title"].values,
    "PageRank": articles_df.nlargest(10, "pagerank")["title"].values,
}, index=range(1, 11))

in_all = set(top10["In-degree"]) & set(top10["PageRank"])
print("In the top 10 for all three measures:", in_all)
top10

In the top 10 for all three measures: {'Attention deficit hyperactivity disorder', 'Bipolar disorder', 'Autism', 'Psychoanalysis', 'Post-traumatic stress disorder', 'Schizophrenia', 'Cognitive behavioral therapy', 'Major depressive disorder', 'Psychosis'}


,In-degree,PageRank
1,Cognitive behavioral therapy,Cognitive behavioral therapy
2,Major depressive disorder,Major depressive disorder
3,Post-traumatic stress disorder,Post-traumatic stress disorder
4,Bipolar disorder,Schizophrenia
5,Schizophrenia,Psychoanalysis
6,Psychoanalysis,Bipolar disorder
7,Attention deficit hyperactivity disorder,Attention deficit hyperactivity disorder
8,Mood disorder,Psychosis
9,Autism,Autism
10,Psychosis,Obsessive–compulsive disorder


In [ ]:
# links between disorders and therapies
types = dict(zip(articles_df["title"], articles_df["type"]))
links_df["source_type"] = links_df["source"].map(types)
links_df["target_type"] = links_df["target"].map(types)

print("Number of links (rows = from, columns = to):")
display(pd.crosstab(links_df["source_type"], links_df["target_type"], margins=True))
print("Share of each type's outgoing links:")
pd.crosstab(links_df["source_type"], links_df["target_type"], normalize="index").round(2)

Number of links (rows = from, columns = to):


target_type,disorder,therapy,All
source_type,,,
disorder,1943,371,2314
therapy,323,738,1061
All,2266,1109,3375


Share of each type's outgoing links:


target_type,disorder,therapy
source_type,,
disorder,0.84,0.16
therapy,0.30,0.70


In [ ]:
# disorders that link to no therapy

to_therapy = links_df[(links_df["source_type"] == "disorder") &
                      (links_df["target_type"] == "therapy")].groupby("source").size()

disorders = articles_df[articles_df["type"] == "disorder"].copy()
disorders["links_to_therapies"] = disorders["title"].map(to_therapy).fillna(0).astype(int)

no_therapy = disorders[disorders["links_to_therapies"] == 0]
print(f"{len(no_therapy)} of {len(disorders)} disorder articles link to no therapy articles")
no_therapy.nlargest(10, "pageviews_12mo")[["title", "pageviews_12mo", "in_degree"]]

98 of 219 disorder articles link to no therapy articles


,title,pageviews_12mo,in_degree
284,Parkinson's disease,1123721,28
221,Chronic traumatic encephalopathy,883755,4
241,Down syndrome,815904,9
226,Dyslexia,641837,10
243,Frontotemporal dementia,535099,8
397,Exhibitionism,468014,2
271,Catatonia,447242,15
305,Intellectual disability,409857,9
359,Narcolepsy,398861,9
324,Transmissible spongiform encephalopathy,372910,1
